# Structured Tools

The `structured.py` module defines `StructuredTool`, a LangChain tool that can accept and validate multiple named input arguments.

A structured tool may wrap a synchronous function, an asynchronous coroutine, or both. Its input arguments are described through a Pydantic model or JSON Schema dictionary.

# StructuredTool

`StructuredTool` wraps a Python function or coroutine whose input contains one or more structured fields.

## Bases

- `BaseTool`

## Attributes

1. `description`: Stores a description of the tool and its purpose.
   * **Type:**
     ```python
     description: str = ""
     ```

2. `args_schema`: Stores the schema used to validate the tool's input arguments.
   * **Type:**
     ```python
     args_schema: Annotated[
         ArgsSchema,
         SkipValidation()
     ] = Field(
         ...,
         description="The tool schema."
     )
     ```

3. `func`: Stores the synchronous function executed by the tool.
   * **Type:**
     ```python
     func: Callable[..., Any] | None = None
     ```

4. `coroutine`: Stores the asynchronous function executed by the tool.
   * **Type:**
     ```python
     coroutine: Callable[
         ...,
         Awaitable[Any]
     ] | None = None
     ```

### Methods

1. `ainvoke`: Executes the structured tool asynchronously through the Runnable interface.

   When no coroutine is configured, the synchronous `invoke` method is executed in an executor. Otherwise, asynchronous execution is delegated to `BaseTool`.

   * **Syntax:**
     ```python
     async ainvoke(
         self,
         input: str | dict[str, Any] | ToolCall, # Tool input or complete tool call
         config: RunnableConfig | None = None, # Runtime configuration
         **kwargs: Any # Additional execution arguments
     ) -> Any
     ```

2. `_run`: Executes the configured synchronous function.

   When the function accepts a `callbacks` parameter, a child callback manager is injected. When the function accepts a Runnable configuration parameter, the current configuration is also injected.

   A `NotImplementedError` is raised when no synchronous function is configured.

   * **Syntax:**
     ```python
     _run(
         self,
         *args: Any, # Positional arguments passed to the function
         config: RunnableConfig, # Runtime configuration
         run_manager: CallbackManagerForToolRun | None = None, # Synchronous callback manager
         **kwargs: Any # Keyword arguments passed to the function
     ) -> Any
     ```

3. `_arun`: Executes the configured asynchronous coroutine.

   When the coroutine accepts a `callbacks` parameter, a child asynchronous callback manager is injected. When it accepts a Runnable configuration parameter, the current configuration is also injected.

   When no coroutine is configured, execution is delegated to the default asynchronous implementation, which runs `_run` in a separate thread.

   * **Syntax:**
     ```python
     async _arun(
         self,
         *args: Any, # Positional arguments passed to the coroutine
         config: RunnableConfig, # Runtime configuration
         run_manager: AsyncCallbackManagerForToolRun | None = None, # Asynchronous callback manager
         **kwargs: Any # Keyword arguments passed to the coroutine
     ) -> Any
     ```

4. `from_function`: Creates a `StructuredTool` from a synchronous function, an asynchronous coroutine, or both.

   The tool name defaults to the source function's name. The input schema can be supplied explicitly or inferred from the function signature.

   When no explicit description is supplied, the function docstring is used. If schema inference and docstring parsing are enabled, Google-style parameter descriptions may be added to the generated schema.

   A `ValueError` is raised when neither a function nor coroutine is supplied, or when no description can be determined. A `TypeError` is raised when `args_schema` is neither a Pydantic model nor a dictionary.

   * **Syntax:**
     ```python
     @classmethod
     from_function(
         cls,
         func: Callable[..., Any] | None = None, # Synchronous function to wrap
         coroutine: Callable[
             ...,
             Awaitable[Any]
         ] | None = None, # Asynchronous function to wrap
         name: str | None = None, # Optional tool name
         description: str | None = None, # Optional tool description
         return_direct: bool = False, # Whether to stop the agent loop after execution
         args_schema: ArgsSchema | None = None, # Explicit input schema
         infer_schema: bool = True, # Whether to infer the schema from the function
         *,
         response_format: Literal[
             "content",
             "content_and_artifact"
         ] = "content", # Interpretation of the tool's return value
         parse_docstring: bool = False, # Whether to parse Google-style argument descriptions
         error_on_invalid_docstring: bool = False, # Whether invalid docstrings raise an error
         **kwargs: Any # Additional StructuredTool fields
     ) -> StructuredTool
     ```